# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ronitjoshi7/flyrank-ml-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page (one pseudonymized content item) with its totals over a trailing 90-day window: impressions, clicks, sessions, and so on. The file has 30,000 rows from 32 clients. The starter filter (some impressions, at least 90 days old) keeps all 30,000.

Windows, as I understand them from the dataset guide:
   90-day totals (features)
                      
  The label compares the last 30 days with the previous 30 days.


In [2]:
# This cell is for CODE (numbers, a query, a check).
import os
import pandas as pd

path = next((p for p in ["data/raw/content_refresh_anonymized.csv",
                         "content_refresh_anonymized.csv"] if os.path.exists(p)), None)
if path is None:
    raise FileNotFoundError("Upload content_refresh_anonymized.csv using the folder icon in Colab.")
df = pd.read_csv(path)

# (loader above)
print("rows, cols:", df.shape, "| clients:", df["client_id"].nunique())
print("content_id unique:", df["content_id"].is_unique)
print("duplicate (client, content) pairs:", df.duplicated(["client_id", "content_id"]).sum())
print("date-like columns:", [c for c in df.columns if "date" in c.lower() or c.endswith("_at")])
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


rows, cols: (30000, 44) | clients: 32
content_id unique: True
duplicate (client, content) pairs: 0
date-like columns: ['days_since_last_update']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Label / proxy (4 source columns plus the label I create): is_declining_label is 1 when trend_direction is "down". It is a rule-based proxy, not an observed future outcome. It is computed from trend_pct, which is computed exactly from impressions_last_30d and impressions_prev_30d, so all four are never features: trend_direction, trend_pct, impressions_last_30d, impressions_prev_30d.

Features (27): search_volume, competition, cpc, word_count, char_count, impressions_90d, clicks_90d, sessions_90d, ai_sessions_90d, days_with_impressions, days_with_sessions, content_age_days, days_since_last_update, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, plus the tiers competition_level, content_type, main_intent, age_tier, freshness_tier, word_count_tier, char_count_tier, impression_tier, position_tier. Caveat: these are measured totals or buckets of them, not computed from the label. But the 90-day totals include the last 30 days the label compares, so they are not strictly "known before" it. I haven't tested how much label information they carry.

Context (7), for grouping or reading only: content_id and client_id (scrambled codes, used to group and split, never as inputs), pageviews_90d, users_90d, engaged_sessions_90d, scroll_events_90d (close cousins of sessions that I may add later), age_tier_order (sort order of age_tier).

Excluded (6), with why:
- clicks_last_30d, sessions_last_30d, clicks_prev_30d, sessions_prev_30d: same last-30 vs previous-30 window the label compares. I'm holding them out until I separate features from the label window.
- provider_used, model_used: 72% and 19% missing, and I'm not sure what they mean. I'll check the data dictionary before using them.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# (loader above)
features_num = ["search_volume", "competition", "cpc", "word_count", "char_count",
                "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
                "days_with_impressions", "days_with_sessions", "content_age_days",
                "days_since_last_update", "ctr", "avg_position", "engagement_rate",
                "scroll_rate", "ai_traffic_pct"]
features_cat = ["competition_level", "content_type", "main_intent", "age_tier",
                "freshness_tier", "word_count_tier", "char_count_tier",
                "impression_tier", "position_tier"]
label_source = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"]
context = ["content_id", "client_id", "pageviews_90d", "users_90d",
           "engaged_sessions_90d", "scroll_events_90d", "age_tier_order"]
excluded = ["clicks_last_30d", "sessions_last_30d", "clicks_prev_30d", "sessions_prev_30d",
            "provider_used", "model_used"]

buckets = {"feature": features_num + features_cat, "label/proxy": label_source,
           "context": context, "excluded": excluded}
for name, cols in buckets.items():
    print(name, len(cols))
sorted_cols = sum(buckets.values(), [])
print("sorted:", len(sorted_cols), "| unique:", len(set(sorted_cols)), "| file has:", df.shape[1])
print("unsorted columns:", set(df.columns) - set(sorted_cols))

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


feature 27
label/proxy 4
context 7
excluded 6
sorted: 44 | unique: 44 | file has: 44
unsorted columns: set()


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Claim 1, grain: one row per page. Checked: content_id is unique and there are 0 duplicate (client, content) pairs.
Claim 2, counts: 30,000 rows and 32 clients, but uneven. Rows per client range from 3 to 7,008 (median 567), and 8 clients have fewer than 100 rows.
Claim 3, missing values: they follow content type, not chance. search_volume is 100% missing for "feedly article" pages and about 1% for keyword articles. word_count is about 28% missing for keyword articles and 0% for the others. avg_position is 0 for 1,205 pages, which I treat as "no data," not a real rank.
Claim 4, windows: last 30 days plus the previous 30 days never exceeds the 90-day impressions (0 violations), so the windows nest as expected. This also means the 90-day features and the label share the same time period.
Claim 5, leakage: trend_pct matches the impressions change to within 0.05, and "down" matches trend_pct below -20 in 99.99% of rows, so those four columns sit in the label bucket and never become features.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# (loader above)
rpc = df.groupby("client_id").size()
print("rows per client: min", rpc.min(), "| median", int(rpc.median()), "| max", rpc.max())
print("clients under 100 rows:", (rpc < 100).sum())

print(df[["search_volume", "word_count"]].isna().groupby(df["content_type"]).mean().round(2))
print("avg_position == 0:", (df["avg_position"] == 0).sum())

s = df["impressions_last_30d"] + df["impressions_prev_30d"]
print("last30 + prev30 > 90d impressions:", (s > df["impressions_90d"]).sum())

ok = df["impressions_prev_30d"] > 0
calc = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"] * 100
print("max gap, trend_pct vs impressions change:", round((df.loc[ok, "trend_pct"] - calc[ok]).abs().max(), 2))
print("'down' == (trend_pct < -20):", round(((df["trend_pct"] < -20) == (df["trend_direction"] == "down")).mean(), 4))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


rows per client: min 3 | median 567 | max 7008
clients under 100 rows: 8
                    search_volume  word_count
content_type                                 
comparison article           0.00        0.00
feedly article               1.00        0.00
keyword article              0.01        0.28
avg_position == 0: 1205
last30 + prev30 > 90d impressions: 0
max gap, trend_pct vs impressions change: 0.05
'down' == (trend_pct < -20): 0.9999


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- No calendar dates in the file, so I can't say which period the 90 days cover or check for seasonality.
- The label and the features share the same time period. impressions_90d includes the last 30 days that the label compares. So my results describe this proxy label, not a forecast of the future.
- The label is a rule, and trend_pct is missing for "new" and "flat" pages (11% of rows).
- Clients are very uneven (3 to 7,008 rows, 8 clients under 100), so large clients could dominate. I need to split by client_id so no client appears in both train and test.
- Missing values follow content type, so filling them with 0 would invent a signal. I'll add "was missing" flags instead.
- avg_position = 0 means no data (1,205 rows). Rates are percentages, and scroll_rate and ai_traffic_pct go above 100 (up to 300 in this file), likely because they come from different measurement systems.
- AI sessions are sparse: only 1,930 of 30,000 pages have any, so I won't build conclusions on them.
- IDs are scrambled, so I can't check real pages. Everything here is observational, not causal.
- If I move to the warehouse later, I'd need to check each client's tracking start date, since early rows have search data only and GA4 columns filled with zeros. I haven't opened the warehouse, so this comes from the dataset guide, not from a check.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# (loader above)
print("date-like columns:", [c for c in df.columns if "date" in c.lower() or c.endswith("_at")])
rpc = df.groupby("client_id").size()
print("rows per client: min", rpc.min(), "| max", rpc.max(), "| clients under 100:", (rpc < 100).sum())
print("trend_pct missing %:", round(df["trend_pct"].isna().mean() * 100, 1))
print("avg_position == 0:", (df["avg_position"] == 0).sum())
print("max scroll_rate, ai_traffic_pct:", round(df["scroll_rate"].max(), 1), round(df["ai_traffic_pct"].max(), 1))
print("pages with any AI sessions:", (df["ai_sessions_90d"] > 0).sum(), "of", len(df))

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


date-like columns: ['days_since_last_update']
rows per client: min 3 | max 7008 | clients under 100: 8
trend_pct missing %: 11.3
avg_position == 0: 1205
max scroll_rate, ai_traffic_pct: 300.0 300.0
pages with any AI sessions: 1930 of 30000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.